# Performance low flows

In [ ]:
# general
from scipy.stats import spearmanr
from pathlib import Path
import pandas as pd
import yaml 
import os

# plotting
import plotly.graph_objects as go
from   plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler

wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes/')
os.chdir(wd)

# import config and local module
with open("config.yml") as f:
    cfg = yaml.safe_load(f)

## Data

In [ ]:
# metrics 
metrics_droughts    = pd.read_csv("data/hydro/performance_lowflows.csv", index_col=0)
metrics_performance = pd.read_csv("data/hydro/performance_metrics.csv", index_col=0)

# andeanGC metadata
andeanGC_metadata = pd.read_csv(Path(cfg["dir"]["andeangc"]) / "dataset/AndeanGC_metadata.csv", index_col=0)
andeanGC_metadata = andeanGC_metadata.join(metrics_droughts, how="left").dropna(axis =0)
andeanGC_metadata = andeanGC_metadata.join(metrics_performance.split, how="left").dropna(axis =0)

## Plot

In [ ]:
# Create subplots
fig = make_subplots(rows=2, cols=3, horizontal_spacing=0.07, vertical_spacing=0.1)

# Define styling parameters
colorscale = [(0, '#fde725'), (0.1, '#6ece58'), (0.2, '#35b779'), (0.6, '#31688e'), (1, '#440154')]
scaler = MinMaxScaler(feature_range=(7, 25))
line_train = "grey"
line_validation = "#0c0c0c"
opacity = 0.9

# Helper function to add scatter plots
def add_scatter_plot(x_data, y_data, data_type, row, col, show_colorbar=False):
    dataset = andeanGC_metadata[andeanGC_metadata.split == data_type]
    line_color = line_train if data_type == 'train' else line_validation
    
    fig.add_trace(go.Scatter(
        x=dataset[x_data], y=dataset[y_data],
        mode='markers', name=data_type.capitalize(), opacity=opacity, showlegend=False,
        marker=dict(
            size=scaler.fit_transform(dataset[['basin_area']]).flatten(),
            color=dataset['glacier_area_RGI60'], 
            cmax=25, cmin=0, showscale=show_colorbar,
            colorbar=dict(title="Glacier<br>area", x=1.02, thickness=18, ticksuffix="%"),
            colorscale=colorscale, line_width=1, line_color=line_color
        )
    ), row=row, col=col)

# Helper function to add 1:1 reference line
def add_reference_line(range_vals, row, col):
    fig.add_trace(go.Scatter(
        x=range_vals, y=range_vals, mode='lines',
        line=dict(color="#BAC7DA", dash='dot'), 
        showlegend=False
    ), row=row, col=col)

# Plot configurations
plots = [
    {'x': 'Q90_obs', 'y': 'Q90_sim', 'range': [-0.5, 6], 'dtick': 2, 'row': 1, 'col': 1,
     'xlabel': 'Q<sub>90</sub> (mm d<sup>-1</sup>) <span style="color:lightgrey">[obs]</span>', 
     'ylabel': 'Q<sub>90</sub> (mm d<sup>-1</sup>) <span style="color:lightgrey">[sim]</span>'},
    {'x': 'min_30day_obs', 'y': 'min_30day_sim', 'range': [-0.5, 5], 'dtick': 1, 'row': 1, 'col': 2,
     'xlabel': 'Min 30-day flow (mm d<sup>-1</sup>) <span style="color:lightgrey">[obs]</span>', 
     'ylabel': 'Min 30-day flow (mm d<sup>-1</sup>) <span style="color:lightgrey">[sim]</span>'},
    {'x': 'doy_min_obs', 'y': 'doy_min_sim', 'range': [0, 365], 'dtick': 60, 'row': 2, 'col': 1,
     'xlabel': 'DOY of minimum flow (julian day) <span style="color:lightgrey">[obs]</span>', 
     'ylabel': 'DOY of minimum flow (julian day) <span style="color:lightgrey">[sim]</span>'},
    {'x': 'intensity_obs', 'y': 'intensity_sim', 'range': [-0.1, 1.2], 'dtick': 0.3, 'row': 2, 'col': 2,
     'xlabel': 'Drought intensity (-) <span style="color:lightgrey">[obs]</span>', 
     'ylabel': 'Drought intensity (-) <span style="color:lightgrey">[sim]</span>'},
    {'x': 'severity_obs', 'y': 'severity_sim', 'range': [-100, 900], 'dtick': 300, 'row': 1, 'col': 3,
     'xlabel': 'Drought severity (mm) <span style="color:lightgrey">[obs]</span>', 
     'ylabel': 'Drought severity (mm) <span style="color:lightgrey">[sim]</span>'},
    {'x': 'events_per_year_obs', 'y': 'events_per_year_sim', 'range': [0, 10], 'dtick': 2, 'row': 2, 'col': 3,
     'xlabel': 'Drought events per year (-) <span style="color:lightgrey">[obs]</span>', 
     'ylabel': 'Drought events per year (-) <span style="color:lightgrey">[sim]</span>'},
]

# Create all plots
for i, plot in enumerate(plots):
    add_reference_line(plot['range'], plot['row'], plot['col'])
    add_scatter_plot(plot['x'], plot['y'], 'train', plot['row'], plot['col'])
    add_scatter_plot(plot['x'], plot['y'], 'validation', plot['row'], plot['col'])
    add_scatter_plot(plot['x'], plot['y'], 'test', plot['row'], plot['col'], 
                    show_colorbar=(i == 0))  # Only show colorbar on first plot
    
    fig.update_xaxes(title_text=plot['xlabel'], range=plot['range'],
                    dtick=plot['dtick'], row=plot['row'], col=plot['col'])
    fig.update_yaxes(title_text=plot['ylabel'], range=plot['range'],
                    dtick=plot['dtick'], row=plot['row'], col=plot['col'])

# Calculate Spearman correlation for each plot
correlations = []
for plot in plots:
    r_train, _ = spearmanr(andeanGC_metadata[andeanGC_metadata.split == 'train'][plot['x']], 
                           andeanGC_metadata[andeanGC_metadata.split == 'train'][plot['y']])
    r_validation, _ = spearmanr(andeanGC_metadata[andeanGC_metadata.split != 'train'][plot['x']], 
                                andeanGC_metadata[andeanGC_metadata.split != 'train'][plot['y']])
    correlations.append({'train': r_train, 'validation': r_validation})

# Add legend and correlation annotations with correct subplot locations
annotations = [
    dict(
        x=0.01, y=0.99, xref="paper", yref="paper",
        text="<br><span style='color:black'>○</span> Validation / Test (VT)<br><span style='color:lightgrey'>○</span> Train (T)",
        showarrow=False, font=dict(size=16), bgcolor="rgba(255,255,255,0.55)",
        borderwidth=0, align="left"
    ),
    dict(
        x=0.15, y=0.57, xref="paper", yref="paper",
        text=f"r = {correlations[0]['train']:.2f} (T) / {correlations[0]['validation']:.2f} (VT)",
        showarrow=False, font=dict(size=13), bgcolor="rgba(255,255,255,0.7)", borderwidth=0
    ),
    dict(
        x=0.57, y=0.57, xref="paper", yref="paper",
        text=f"r = {correlations[1]['train']:.2f} (T) / {correlations[1]['validation']:.2f} (VT)",
        showarrow=False, font=dict(size=13), bgcolor="rgba(255,255,255,0.7)", borderwidth=0
    ),
    dict(
        x=0.99, y=0.57, xref="paper", yref="paper",
        text=f"r = {correlations[2]['train']:.2f} (T) / {correlations[2]['validation']:.2f} (VT)",
        showarrow=False, font=dict(size=13), bgcolor="rgba(255,255,255,0.7)", borderwidth=0
    ),
    dict(
        x=0.15, y=0.01, xref="paper", yref="paper",
        text=f"r = {correlations[3]['train']:.2f} (T) / {correlations[3]['validation']:.2f} (VT)",
        showarrow=False, font=dict(size=13), bgcolor="rgba(255,255,255,0.7)", borderwidth=0
    ),
    dict(
        x=0.57, y=0.01, xref="paper", yref="paper",
        text=f"r = {correlations[4]['train']:.2f} (T) / {correlations[4]['validation']:.2f} (VT)",
        showarrow=False, font=dict(size=13), bgcolor="rgba(255,255,255,0.7)", borderwidth=0
    ),
    dict(
        x=0.99, y=0.01, xref="paper", yref="paper",
        text=f"r = {correlations[5]['train']:.2f} (T) / {correlations[5]['validation']:.2f} (VT)",
        showarrow=False, font=dict(size=13), bgcolor="rgba(255,255,255,0.7)", borderwidth=0
    )
]

# Update layout
fig.update_xaxes(ticks="outside", griddash="dot", zeroline=False, title_standoff=2)
fig.update_yaxes(ticks="outside", griddash="dot", zeroline=False, title_standoff=2)
fig.update_layout(
    height=700, width=1100, font=dict(size=14, family ="Arial"), annotations=annotations,
    template="plotly", plot_bgcolor="rgba(229,236,246,0.5)",
    margin=dict(t=10, b=50, l=50, r=10)
)

fig.write_image("figures/fig_performance_droughts.svg",  height=700, width=1100)
fig.show()